# Differential expression analysis


You have run the nf-core/rnaseq pipeline and checked the first quality control metrics of your fastq files. This was, however, only the primary analysis and we want to take it further.

Due to the computational demand of the pipeline, you only ran the pipeline on two of the 16 samples in the study yesterday. We provide you an essential output of nf-core/rnaseq pipeline in the `data` folder: It contains the combined epression matrix as produced by Salmon, which provides transcript levels for each gene (rows) and each sample (columns).


We would now like to understand exactly the difference between the expression in our groups of mice. 
Which pipeline would you use for this?

nf-core/differentialabundance pipeline

Have a close look at the pipeline's "Usage" page on the [nf-core docs](nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

In [6]:
import pandas as pd

matrix_file = "data/salmon.merged.gene_counts.tsv"

df_matrix = pd.read_csv(matrix_file, sep="\t")
sample_ids = list(df_matrix.columns[2:])

samplesheet = pd.DataFrame({"sample": sample_ids})
samplesheet["Condition"] = samplesheet["sample"].str.rsplit("_", n=1).str[0]

samplesheet.to_csv("samplesheet.csv", index=False)
print(samplesheet)

        sample Condition
0   Sham_oxy_1  Sham_oxy
1   Sham_oxy_2  Sham_oxy
2   Sham_oxy_3  Sham_oxy
3   Sham_oxy_4  Sham_oxy
4   Sham_Sal_1  Sham_Sal
5   Sham_Sal_2  Sham_Sal
6   Sham_Sal_3  Sham_Sal
7   Sham_Sal_4  Sham_Sal
8    SNI_oxy_1   SNI_oxy
9    SNI_oxy_2   SNI_oxy
10   SNI_oxy_3   SNI_oxy
11   SNI_oxy_4   SNI_oxy
12   SNI_Sal_1   SNI_Sal
13   SNI_Sal_2   SNI_Sal
14   SNI_Sal_3   SNI_Sal
15   SNI_Sal_4   SNI_Sal


Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [ ]:

nextflow run nf-core/differentialabundance \
    -profile rnaseq,docker \
    --input ./day_03/samplesheet.csv \
    --contrasts ./day_03/data/contrasts.csv \
    --matrix ./day_03/data/salmon.merged.gene_counts.tsv \
    --run_gsea true \
    --gtf ./day_03/Mus_musculus.GRCm38.102.gtf/Mus_musculus.GRCm38.102.gtf \
    --outdir ./day_03/results

Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

--contrasts ./day_03/data/contrasts.csv: provided file that defines which groups are compared (variable, reference, target), here oxycodone vs. saline within SNI and within Sham
--matrix ./day_03/data/salmon.merged.gene_counts.tsv: raw gene count matrix from nf-core/rnaseq (Salmon). I used raw counts because DESeq2 does its own normalisation
--run_gsea true: additionally runs gene set enrichment analysis, to see which pathways shift between the groups.
--gtf ./day_03/Mus_musculus.GRCm38.102.gtf/Mus_musculus.GRCm38.102.gtf: mouse gene annotation, used to annotate genes (names, IDs) and for GSEA
--outdir ./day_03/results: folder for all results

I created samplesheet.csv with pandas from the column names of the matrix, and it contains the group of each sample in the column Condition. The contrast file was provided and tells the pipeline which groups to compare. The GTF file is the reference annotation.

What were the outputs of the pipeline?

-report: an HTML report (study_differentialabundance_report.html) that summarises the whole analysis, plus a zipped copy (study.zip)
-plots/exploratory/: quality control plots of the samples (boxplot, density plot, MAD correlation, 2D and 3D PCA, sample dendrogram). They show how similar the samples are and help to find outliers
-plots/differential/: plots for each contrast, e.g. volcano plots (one folder per contrast: condition_control_treated_study and condition_control_treated_test_study)
-plots/qc/: DESeq2 dispersion plots for each contrast
-tables/: result tables, including normalised counts and the differential expression results per contrast (log2 fold changes, p-values, adjusted p-values)



Would you exclude any samples? If yes, which and why?



Yes, I would consider excluding SNI_Sal_2 and SNI_Sal_4. They lie far from all other samples in the PCA and form a separate branch in the dendrogram, and both show unusually high values in the boxplots. The MAD plot does not flag them, because the two resemble each other and shift the group median. Since excluding both would leave only two SNI_Sal replicates, I would first check their quality metrics and exclude them only if a technical problem is found.

How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

With the BH-adjusted threshold (padj ≤ 0.1), 7 genes were differentially expressed in Sham oxy vs. Sham Sal (all up) and 16 in SNI oxy vs. SNI Sal (1 up, 15 down).
The paper reports different numbers of differentially expressed genes in the NAc, so our this results do not confirm it. Likely reasons are the different processing (STAR/Salmon instead of HISAT2/HTSeq), the paper’s looser cutoff (nominal p < 0.05 without multiple-testing correction, which lets many false positives pass), and the high variability of the SNI_Sal group. Only the general picture, that oxycodone withdrawal changes gene expression in the NAc, is plausible. This illustrates the limited reproducibility of the published analysis.

The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

Nucleus accumbens (NAc): a core region of the brain’s reward system, involved in motivation, reward processing and addiction
Ventral tegmental area (VTA): contains dopamine-producing neurons that project to the NAc and the prefrontal cortex, so it is a key source of the dopamine signal in the reward pathway
Medial prefrontal cortex (mPFC): involved in decision making and executive control, and it plays an important role in addiction and relapse

Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

No not findable in the paper.

Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state? 

No, a differentially expressed genes list alone would not be sufficient. Biologists need to know what the changes mean, not just which genes have changed. Moreover, if a broad threshold is used, such a list would also contain false positives. The publication therefore builds on the list: A signaling pathway analysis (IPA) identifies HDAC1 as a likely upstream regulator in the NAc and mPFC, and this prediction is subsequently tested using the HDAC1/HDAC2 inhibitor RBC1HI, which alleviates withdrawal behavior, particularly in mice with nerve injuries. The DEGs thus serve more as a starting point for interpretation than as the final result.




Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.